# Drug Discovery Workflow

This notebook demonstrates molecule generation, docking, and ADMET prediction for cancer drug discovery.

In [ ]:
import sys
sys.path.append('..')

from drug_discovery import MoleculeGenerator, MolecularDocking, ADMETPredictor
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## Generate Candidate Molecules

In [ ]:
generator = MoleculeGenerator()

# Generate molecules for EGFR target
molecules = generator.generate_molecules(
    target_protein="EGFR",
    num_molecules=20,
    method="transformer"
)

print(f"Generated {len(molecules)} candidate molecules")
print("\nExample SMILES:")
for i, smiles in enumerate(molecules[:5], 1):
    print(f"{i}. {smiles}")

## Calculate Molecular Properties

In [ ]:
# Calculate properties for all molecules
properties_list = []

for smiles in molecules:
    props = generator.calculate_properties(smiles)
    props['smiles'] = smiles
    properties_list.append(props)

properties_df = pd.DataFrame(properties_list)
print(f"Calculated properties for {len(properties_df)} molecules\n")
display(properties_df.head())

In [ ]:
# Visualize molecular properties
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].hist(properties_df['molecular_weight'], bins=15, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Molecular Weight Distribution')
axes[0, 0].set_xlabel('MW (Da)')
axes[0, 0].axvline(500, color='red', linestyle='--', label='Lipinski limit')
axes[0, 0].legend()

axes[0, 1].hist(properties_df['logP'], bins=15, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('LogP Distribution')
axes[0, 1].set_xlabel('LogP')
axes[0, 1].axvline(5, color='red', linestyle='--', label='Lipinski limit')
axes[0, 1].legend()

axes[1, 0].scatter(properties_df['molecular_weight'], properties_df['logP'], alpha=0.6)
axes[1, 0].set_title('MW vs LogP')
axes[1, 0].set_xlabel('Molecular Weight (Da)')
axes[1, 0].set_ylabel('LogP')

drug_like_count = properties_df['drug_like'].sum()
axes[1, 1].bar(['Drug-like', 'Non-drug-like'], 
               [drug_like_count, len(properties_df) - drug_like_count],
               color=['green', 'orange'])
axes[1, 1].set_title('Drug-likeness')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Molecular Docking

In [ ]:
docker = MolecularDocking()

# Dock top 10 drug-like molecules
drug_like_molecules = properties_df[properties_df['drug_like'] == True]['smiles'].tolist()

docking_results = []
for smiles in drug_like_molecules[:10]:
    result = docker.dock_molecule(
        ligand_smiles=smiles,
        protein_pdb="../data/processed/structures/EGFR_predicted.pdb"
    )
    docking_results.append(result)

print(f"Docked {len(docking_results)} molecules")

In [ ]:
# Rank compounds
ranked_compounds = docker.rank_compounds(docking_results)

print("Top 5 Compounds by Binding Affinity:\n")
display(ranked_compounds[['ligand', 'binding_affinity', 'ligand_efficiency']].head())

In [ ]:
# Visualize docking results
plt.figure(figsize=(10, 6))
plt.barh(range(len(ranked_compounds)), ranked_compounds['binding_affinity'], color='coral')
plt.xlabel('Binding Affinity (kcal/mol)')
plt.ylabel('Compound Rank')
plt.title('Binding Affinity of Docked Compounds')
plt.axvline(-7.0, color='green', linestyle='--', label='Good binder threshold')
plt.legend()
plt.tight_layout()
plt.show()

## ADMET Prediction

In [ ]:
admet_predictor = ADMETPredictor()

# Analyze top compound
top_compound = ranked_compounds.iloc[0]['ligand']

print(f"Analyzing top compound: {top_compound}\n")

# Predict ADMET
admet = admet_predictor.predict_admet(top_compound)

print("ADMET Profile:")
print("\nAbsorption:")
print(f"  Caco-2 permeability: {admet['absorption']['caco2_permeability']}")
print(f"  Bioavailability: {admet['absorption']['bioavailability']:.1%}")

print("\nDistribution:")
print(f"  BBB penetration: {admet['distribution']['blood_brain_barrier']}")
print(f"  Plasma protein binding: {admet['distribution']['plasma_protein_binding']:.1%}")

print("\nMetabolism:")
print(f"  CYP450 substrates: {', '.join(admet['metabolism']['cyp450_substrate'])}")

print("\nExcretion:")
print(f"  Half-life: {admet['excretion']['half_life_hours']:.1f} hours")
print(f"  Clearance: {admet['excretion']['clearance']}")

print("\nToxicity:")
print(f"  Hepatotoxicity: {admet['toxicity']['hepatotoxicity']}")
print(f"  Cardiotoxicity: {admet['toxicity']['cardiotoxicity']}")
print(f"  Mutagenicity: {admet['toxicity']['mutagenicity']}")

In [ ]:
# Check drug-likeness
drug_like = admet_predictor.check_drug_likeness(top_compound)

print("\nDrug-likeness Assessment:")
print(f"  Status: {drug_like['assessment']}")
print(f"  Lipinski violations: {drug_like['lipinski_violations']}")
print(f"\nMolecular Properties:")
for prop, value in drug_like['properties'].items():
    print(f"  {prop}: {value}")

## Summary and Recommendations

In [ ]:
print("Drug Discovery Pipeline Summary:\n")
print(f"Total molecules generated: {len(molecules)}")
print(f"Drug-like molecules: {drug_like_count}")
print(f"Molecules docked: {len(docking_results)}")
print(f"\nTop Candidate:")
print(f"  SMILES: {top_compound}")
print(f"  Binding affinity: {ranked_compounds.iloc[0]['binding_affinity']:.2f} kcal/mol")
print(f"  Ligand efficiency: {ranked_compounds.iloc[0]['ligand_efficiency']:.2f}")
print(f"  Drug-like: {drug_like['is_drug_like']}")
print(f"  Bioavailability: {admet['absorption']['bioavailability']:.1%}")
print(f"\nRecommendation: {'Proceed to experimental validation' if drug_like['is_drug_like'] else 'Optimize molecule further'}")